# Home Loan Default Risk Prediction
**Student Name:** Kevin Thomas Jacob  
**Date:** March 10, 2026

---

## 1. Project Introduction
The goal of this project is to build a predictive model that determines the likelihood of a loan applicant defaulting on their home equity loan. We use the `home_loan_default_risk.csv` dataset, which contains financial and demographic information for nearly 6,000 applicants.

### Objectives:
* **Cleanse Data:** Address missing values in critical columns like `DEBTINC` and `MORTDUE`.
* **Feature Engineering:** Prepare categorical variables (`REASON`, `JOB`) for machine learning.
* **Model Training:** Implement a **Random Forest Classifier** to handle the complex relationships in the data.
* **Risk Assessment:** Generate probability scores to help financial institutions make informed lending decisions.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

df = pd.read_csv('home_loan_default_risk.csv')

# Initial Data Inspection
print("First 5 rows of data:")
print(df.head())
print("\nDataset Schema and Missing Values:")
print(df.info())

First 5 rows of data:
   BAD  LOAN  MORTDUE     VALUE   REASON     JOB   YOJ  DEROG  DELINQ  \
0    1  1100  25860.0   39025.0  HomeImp   Other  10.5    0.0     0.0   
1    1  1300  70053.0   68400.0  HomeImp   Other   7.0    0.0     2.0   
2    1  1500  13500.0   16700.0  HomeImp   Other   4.0    0.0     0.0   
3    1  1500      NaN       NaN      NaN     NaN   NaN    NaN     NaN   
4    0  1700  97800.0  112000.0  HomeImp  Office   3.0    0.0     0.0   

        CLAGE  NINQ  CLNO  DEBTINC  
0   94.366667   1.0   9.0      NaN  
1  121.833333   0.0  14.0      NaN  
2  149.466667   1.0  10.0      NaN  
3         NaN   NaN   NaN      NaN  
4   93.333333   0.0  14.0      NaN  

Dataset Schema and Missing Values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5960 entries, 0 to 5959
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   BAD      5960 non-null   int64  
 1   LOAN     5960 non-null   int64  
 2   MORTDUE  5442 non

## 2. Data Preprocessing
Real-world financial data often contains missing entries. In this section, we:
1. **Impute Missing Values:** Use the **median** strategy for numerical columns to ensure outlier robustness.
2. **Handle Categorical Data:** Fill missing values in `REASON` and `JOB` with 'Unknown' and apply **Label Encoding**.
3. **Feature Selection:** Select variables such as `LOAN` (amount), `DEBTINC` (debt-to-income ratio), and `DELINQ` (delinquent credit lines) as primary predictors for the target variable `BAD` (1 = Default, 0 = No Default).

In [ ]:
# Handle missing values
imputer = SimpleImputer(strategy='median')
num_cols = ['LOAN', 'MORTDUE', 'VALUE', 'YOJ', 'DEROG', 'DELINQ', 'CLAGE', 'NINQ', 'CLNO', 'DEBTINC']
df[num_cols] = imputer.fit_transform(df[num_cols])

# Encode categorical variables
le = LabelEncoder()
df['REASON'] = df['REASON'].fillna('Unknown')
df['JOB'] = df['JOB'].fillna('Unknown')
df['REASON_ENCODED'] = le.fit_transform(df['REASON'])
df['JOB_ENCODED'] = le.fit_transform(df['JOB'])

# Select features and split data
features = num_cols + ['REASON_ENCODED', 'JOB_ENCODED']
X = df[features]
y = df['BAD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features for consistent model performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Model Training
We selected the **Random Forest Classifier** for this task. Random Forests are particularly effective for financial risk modeling because they are less prone to overfitting and can capture non-linear interactions between variables like income and debt. We use `class_weight='balanced'` to account for the fact that loan defaults are less frequent than successful payments.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# Generate Predictions
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Model Training Complete.")

Model Training Complete.


## 4. Practical Application: Predicting Risk
To demonstrate the model's utility, we simulate a new loan application. By inputting specific financial parameters, the model outputs both a **binary decision** (Approve/Default) and a **probability score**, allowing the bank to set different risk thresholds.

In [ ]:
# Example data for a new applicant
new_applicant_data = pd.DataFrame({
    'LOAN': [15000.0], 'MORTDUE': [50000.0], 'VALUE': [70000.0],
    'YOJ': [5.0], 'DEROG': [0.0], 'DELINQ': [0.0],
    'CLAGE': [100.0], 'NINQ': [1.0], 'CLNO': [10.0],
    'DEBTINC': [30.0], 'REASON_ENCODED': [0], 'JOB_ENCODED': [0]
})

# Preprocess and Predict
new_data_scaled = scaler.transform(new_applicant_data[features])
prediction = model.predict(new_data_scaled)
probability = model.predict_proba(new_data_scaled)[:, 1]

print(f"Default Prediction: {prediction[0]} (1=Default, 0=Safe)")
print(f"Risk Probability: {probability[0]:.2f}")

Default Prediction: 0 (1=Default, 0=Safe)
Risk Probability: 0.02


## 5. Conclusion
The Home Loan Default Risk model provides a robust framework for assessing applicant creditworthiness.

**Key Business Takeaways:**
* **Imbalance Handling:** By using a balanced Random Forest, we improved the model's ability to catch defaults that might otherwise be missed.
* **Risk Probability:** Providing a probability score (e.g., 0.02 for the test applicant) allows the bank to perform more granular risk-based pricing.
* **Future Work:** Further performance gains could be achieved by exploring Gradient Boosting (XGBoost) or fine-tuning hyperparameters via Grid Search.